# Experiment 0.1 — General Comparison

Analysis-only notebook for the finalized Exp0.1 General Comparison artifacts.

Primary systems: `short_mid_long`, `short_mid`, `mid_long`, `(234)->(234) + Fixed250 Linear`, `Raw Fixed250 + Linear`, and `Raw Relative10 + Linear`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_0_1_general_comparison' / 'general_comparison_v1'
RUNS = pd.read_csv(ART / 'runs.csv')
SUMMARY = pd.read_csv(ART / 'summary.csv')
BASELINES = pd.read_csv(ART / 'baseline_results.csv')
COMPARISON = pd.read_csv(ART / 'comparison_summary.csv')
CAPACITY = pd.read_csv(ART / 'paired_capacity_effects.csv')
OBJECTIVE = pd.read_csv(ART / 'paired_objective_effects.csv')
ARCH = pd.read_csv(ART / 'paired_architecture_effects.csv')
with (ART / 'manifest.json').open('r', encoding='utf-8') as handle:
    MANIFEST = json.load(handle)
ART


## Overall comparison

The common table compares every trained SNN system with the two deterministic raw baselines: **Raw Fixed250 + Linear** and **Raw Relative10 + Linear**.

In [ ]:
display(COMPARISON.sort_values('mean_test_ba', ascending=False).reset_index(drop=True))


In [ ]:
plot_df = COMPARISON.sort_values('mean_test_ba', ascending=True).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, max(5, 0.38 * len(plot_df))))
ax.barh(plot_df['system'], plot_df['mean_test_ba'])
ax.set_xlabel('Test balanced accuracy')
ax.set_title('Exp0.1 General Comparison')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()


## Binary vs multi-H hidden communication capacity

Positive values mean hidden event capacity 31 outperforms pure binary hidden spikes.

In [ ]:
capacity_summary = (CAPACITY.groupby(['family', 'architecture', 'objective'])['delta_test_ba_multi_h_minus_binary']
                    .agg(['mean', 'std', 'count']).reset_index())
display(capacity_summary.sort_values('mean', ascending=False))


## Timestep CE vs WholeCount CE

This comparison is restricted to the direct SNN families. Final inference remains Output WholeCount in both cases.

In [ ]:
objective_summary = (OBJECTIVE.groupby(['architecture', 'variant'])['delta_test_ba_timestep_minus_whole_count']
                     .agg(['mean', 'std', 'count']).reset_index())
display(objective_summary.sort_values('mean', ascending=False))


## Temporal hierarchy effects

`add_long_layer`: `short_mid_long - short_mid`.  
`shift_coverage_longer`: `mid_long - short_mid`.

In [ ]:
arch_summary = (ARCH.groupby(['comparison', 'objective', 'variant'])['delta_test_ba']
                .agg(['mean', 'std', 'count']).reset_index())
display(arch_summary.sort_values(['comparison', 'mean'], ascending=[True, False]))


## Raw baseline anchors

In [ ]:
display(BASELINES[['system', 'feature_dim', 'probe_C', 'val_ba', 'test_ba', 'test_accuracy', 'test_macro_f1']])


## Training stability / per-seed inspection

In [ ]:
display(RUNS.sort_values(['family', 'architecture', 'objective', 'variant', 'seed']).reset_index(drop=True))
